# EDA 03 - Key Integrity Checks

This notebook checks primary-key and foreign-key consistency across the core entity tables. It normalizes column names and key values before comparison so casing, whitespace, and numeric/string differences do not create false mismatches.

## 1. Setup and file discovery

Define the expected files and locate them recursively from the current working directory. Missing files are reported but do not stop execution.

In [1]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path.cwd()
EXPECTED_FILES = [
    "products.csv",
    "customers.csv",
    "geography.csv",
    "orders.csv",
    "order_items.csv",
    "returns.csv",
]

all_csv_paths = list(PROJECT_ROOT.rglob("*.csv"))
paths_by_name = {}
for path in all_csv_paths:
    paths_by_name.setdefault(path.name, []).append(path)

file_rows = []
for filename in EXPECTED_FILES:
    matches = paths_by_name.get(filename, [])
    if matches:
        selected = sorted(matches, key=lambda p: (len(p.parts), str(p)))[0]
        file_rows.append({
            "file": filename,
            "status": "found",
            "path": str(selected.relative_to(PROJECT_ROOT)),
            "match_count": len(matches),
        })
    else:
        file_rows.append({
            "file": filename,
            "status": "missing",
            "path": None,
            "match_count": 0,
        })

file_discovery_df = pd.DataFrame(file_rows)
missing_files_df = file_discovery_df[file_discovery_df["status"] == "missing"].copy()

print(f"Project root: {PROJECT_ROOT}")
file_discovery_df

Project root: D:\Code\Datathon 2026


,file,status,path,match_count
0,products.csv,found,datathon-2026-round-1\products.csv,1
1,customers.csv,found,datathon-2026-round-1\customers.csv,1
2,geography.csv,found,datathon-2026-round-1\geography.csv,1
3,orders.csv,found,datathon-2026-round-1\orders.csv,1
4,order_items.csv,found,datathon-2026-round-1\order_items.csv,1
5,returns.csv,found,datathon-2026-round-1\returns.csv,1


## 2. Load CSV files

Load each available file safely. Missing or failed files are captured in the load summary.

In [2]:
loaded = {}
load_rows = []

for _, row in file_discovery_df.iterrows():
    filename = row["file"]
    if row["status"] != "found":
        load_rows.append({"file": filename, "loaded": False, "rows": np.nan, "columns": np.nan, "error": "file missing"})
        continue
    try:
        df = pd.read_csv(PROJECT_ROOT / row["path"], low_memory=False)
        loaded[filename] = df
        load_rows.append({"file": filename, "loaded": True, "rows": len(df), "columns": len(df.columns), "error": ""})
    except Exception as exc:
        load_rows.append({"file": filename, "loaded": False, "rows": np.nan, "columns": np.nan, "error": repr(exc)})

load_summary_df = pd.DataFrame(load_rows)
load_summary_df

,file,loaded,rows,columns,error
0,products.csv,True,2412,8,
1,customers.csv,True,121930,7,
2,geography.csv,True,39948,4,
3,orders.csv,True,646945,8,
4,order_items.csv,True,714669,7,
5,returns.csv,True,39939,7,


## 3. Normalize column names and key values

Normalize column names for robust lookup, then normalize key values by trimming whitespace and comparing IDs/ZIPs as strings. ZIP codes are kept as strings so leading zeros would be preserved if present in the raw data.

In [3]:
def normalize_column_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def build_column_lookup(df):
    return {normalize_column_name(col): col for col in df.columns}


def find_column(df, candidates):
    lookup = build_column_lookup(df)
    normalized_candidates = [normalize_column_name(c) for c in candidates]
    for candidate in normalized_candidates:
        if candidate in lookup:
            return lookup[candidate]
    return None


def normalize_key_series(series, key_type="id"):
    raw = series.copy()
    as_str = raw.astype("string")
    normalized = as_str.str.strip()
    normalized = normalized.mask(normalized.isin(["", "nan", "NaN", "None", "<NA>"]))

    # If pandas loaded integer-like IDs/ZIPs as floats or strings like "123.0", normalize them to "123".
    if key_type in {"id", "zip"}:
        numeric = pd.to_numeric(normalized, errors="coerce")
        integer_like = numeric.notna() & np.isclose(numeric, np.floor(numeric))
        normalized = normalized.mask(integer_like, numeric[integer_like].astype("Int64").astype("string"))

    return normalized


def formatting_diagnostics(series, normalized):
    raw_non_null = series.dropna().astype(str)
    raw_as_str = series.astype("string")
    stripped = raw_as_str.str.strip()
    whitespace_count = int((raw_as_str.notna() & (raw_as_str != stripped)).sum())
    empty_after_strip_count = int((raw_as_str.notna() & stripped.eq("")).sum())
    decimal_like_count = int(raw_non_null.str.match(r"^-?\d+\.0+$").sum())
    leading_zero_count = int(raw_non_null.str.match(r"^0+\d+$").sum())
    normalized_null_count = int(normalized.isna().sum())
    return {
        "raw_missing_count": int(series.isna().sum()),
        "normalized_missing_count": normalized_null_count,
        "values_with_outer_whitespace": whitespace_count,
        "empty_strings_after_strip": empty_after_strip_count,
        "decimal_integer_string_count": decimal_like_count,
        "leading_zero_string_count": leading_zero_count,
    }

required_columns = {
    "orders.csv": {"order_id": ["order_id", "order id", "Order ID"], "zip": ["zip", "zipcode", "postal_code", "postal code"]},
    "order_items.csv": {"order_id": ["order_id", "order id", "Order ID"]},
    "returns.csv": {"product_id": ["product_id", "product id", "Product ID"]},
    "products.csv": {"product_id": ["product_id", "product id", "Product ID"]},
    "customers.csv": {"zip": ["zip", "zipcode", "postal_code", "postal code"]},
    "geography.csv": {"zip": ["zip", "zipcode", "postal_code", "postal code"]},
}

resolved_column_rows = []
resolved_columns = {}
for filename, requirements in required_columns.items():
    resolved_columns[filename] = {}
    df = loaded.get(filename)
    for logical_name, candidates in requirements.items():
        actual_col = find_column(df, candidates) if df is not None else None
        resolved_columns[filename][logical_name] = actual_col
        resolved_column_rows.append({
            "file": filename,
            "logical_key": logical_name,
            "resolved_column": actual_col,
            "found": actual_col is not None,
            "available_columns": ", ".join(df.columns.astype(str)) if df is not None else "file not loaded",
        })

resolved_columns_df = pd.DataFrame(resolved_column_rows)
missing_key_columns_df = resolved_columns_df[~resolved_columns_df["found"]].copy()
resolved_columns_df

,file,logical_key,resolved_column,found,available_columns
0,orders.csv,order_id,order_id,True,"order_id, order_date, customer_id, zip, order_status, payment_method, device_type, order_source"
1,orders.csv,zip,zip,True,"order_id, order_date, customer_id, zip, order_status, payment_method, device_type, order_source"
2,order_items.csv,order_id,order_id,True,"order_id, product_id, quantity, unit_price, discount_amount, promo_id, promo_id_2"
3,returns.csv,product_id,product_id,True,"return_id, order_id, product_id, return_date, return_reason, return_quantity, refund_amount"
4,products.csv,product_id,product_id,True,"product_id, product_name, category, segment, size, color, price, cogs"
5,customers.csv,zip,zip,True,"customer_id, zip, city, signup_date, gender, age_group, acquisition_channel"
6,geography.csv,zip,zip,True,"zip, city, region, district"


## 4. Key integrity check summary

Run the required foreign-key checks and summarize total key values, unique keys, matched keys, orphan keys, and formatting diagnostics.

In [4]:
checks = [
    {
        "check_name": "order_items.order_id -> orders.order_id",
        "child_file": "order_items.csv",
        "child_logical_key": "order_id",
        "parent_file": "orders.csv",
        "parent_logical_key": "order_id",
        "key_type": "id",
    },
    {
        "check_name": "returns.product_id -> products.product_id",
        "child_file": "returns.csv",
        "child_logical_key": "product_id",
        "parent_file": "products.csv",
        "parent_logical_key": "product_id",
        "key_type": "id",
    },
    {
        "check_name": "customers.zip -> geography.zip",
        "child_file": "customers.csv",
        "child_logical_key": "zip",
        "parent_file": "geography.csv",
        "parent_logical_key": "zip",
        "key_type": "zip",
    },
    {
        "check_name": "orders.zip -> geography.zip",
        "child_file": "orders.csv",
        "child_logical_key": "zip",
        "parent_file": "geography.csv",
        "parent_logical_key": "zip",
        "key_type": "zip",
    },
]

summary_rows = []
orphan_samples = {}
formatting_rows = []

for check in checks:
    check_name = check["check_name"]
    child_df = loaded.get(check["child_file"])
    parent_df = loaded.get(check["parent_file"])
    child_col = resolved_columns.get(check["child_file"], {}).get(check["child_logical_key"])
    parent_col = resolved_columns.get(check["parent_file"], {}).get(check["parent_logical_key"])

    if child_df is None or parent_df is None or child_col is None or parent_col is None:
        summary_rows.append({
            "check_name": check_name,
            "status": "skipped_missing_file_or_column",
            "child_file": check["child_file"],
            "child_column": child_col,
            "parent_file": check["parent_file"],
            "parent_column": parent_col,
            "total_child_key_values": np.nan,
            "unique_child_key_values": np.nan,
            "unique_child_keys_existing_in_parent": np.nan,
            "unique_orphan_keys": np.nan,
            "orphan_key_value_count": np.nan,
            "orphan_unique_pct": np.nan,
            "orphan_value_pct": np.nan,
        })
        orphan_samples[check_name] = pd.DataFrame({"message": ["Check skipped because file or column is missing."]})
        continue

    child_keys = normalize_key_series(child_df[child_col], check["key_type"])
    parent_keys = normalize_key_series(parent_df[parent_col], check["key_type"])

    child_non_null = child_keys.dropna()
    parent_non_null = parent_keys.dropna()
    unique_child = pd.Index(child_non_null.unique())
    unique_parent = pd.Index(parent_non_null.unique())
    orphan_unique = unique_child.difference(unique_parent)
    existing_unique_count = len(unique_child.intersection(unique_parent))
    orphan_value_mask = child_keys.isin(orphan_unique)
    orphan_value_count = int(orphan_value_mask.sum())

    child_diag = formatting_diagnostics(child_df[child_col], child_keys)
    parent_diag = formatting_diagnostics(parent_df[parent_col], parent_keys)
    for side, filename, column, diag in [
        ("child", check["child_file"], child_col, child_diag),
        ("parent", check["parent_file"], parent_col, parent_diag),
    ]:
        formatting_rows.append({
            "check_name": check_name,
            "side": side,
            "file": filename,
            "column": column,
            **diag,
        })

    summary_rows.append({
        "check_name": check_name,
        "status": "checked",
        "child_file": check["child_file"],
        "child_column": child_col,
        "parent_file": check["parent_file"],
        "parent_column": parent_col,
        "total_child_key_values": int(child_non_null.shape[0]),
        "unique_child_key_values": int(len(unique_child)),
        "unique_child_keys_existing_in_parent": int(existing_unique_count),
        "unique_orphan_keys": int(len(orphan_unique)),
        "orphan_key_value_count": orphan_value_count,
        "orphan_unique_pct": (len(orphan_unique) / len(unique_child) * 100) if len(unique_child) else 0.0,
        "orphan_value_pct": (orphan_value_count / len(child_non_null) * 100) if len(child_non_null) else 0.0,
    })

    sample_values = pd.Series(orphan_unique[:20], name="orphan_key_sample")
    orphan_samples[check_name] = pd.DataFrame(sample_values) if len(sample_values) else pd.DataFrame({"message": ["No orphan keys detected."]})

key_integrity_summary_df = pd.DataFrame(summary_rows)
formatting_diagnostics_df = pd.DataFrame(formatting_rows)
key_integrity_summary_df

,check_name,status,child_file,child_column,parent_file,parent_column,total_child_key_values,unique_child_key_values,unique_child_keys_existing_in_parent,unique_orphan_keys,orphan_key_value_count,orphan_unique_pct,orphan_value_pct
0,order_items.order_id -> orders.order_id,checked,order_items.csv,order_id,orders.csv,order_id,714669,646945,646945,0,0,0.0,0.0
1,returns.product_id -> products.product_id,checked,returns.csv,product_id,products.csv,product_id,39939,1286,1286,0,0,0.0,0.0
2,customers.zip -> geography.zip,checked,customers.csv,zip,geography.csv,zip,121930,31491,31491,0,0,0.0,0.0
3,orders.zip -> geography.zip,checked,orders.csv,zip,geography.csv,zip,646945,29932,29932,0,0,0.0,0.0


## 5. Detailed orphan-key samples

Show small samples of orphan keys for each required relationship. If none exist, the table states that no orphan keys were detected.

In [5]:
for check_name, sample_df in orphan_samples.items():
    print(check_name)
    display(sample_df)

order_items.order_id -> orders.order_id


,message
0,No orphan keys detected.


returns.product_id -> products.product_id


,message
0,No orphan keys detected.


customers.zip -> geography.zip


,message
0,No orphan keys detected.


orders.zip -> geography.zip


,message
0,No orphan keys detected.


### Key formatting diagnostics

Review missing values and formatting irregularities after normalization. These diagnostics help identify whitespace, decimal-like IDs, empty strings, or leading-zero ZIP values.

In [6]:
formatting_diagnostics_df

,check_name,side,file,column,raw_missing_count,normalized_missing_count,values_with_outer_whitespace,empty_strings_after_strip,decimal_integer_string_count,leading_zero_string_count
0,order_items.order_id -> orders.order_id,child,order_items.csv,order_id,0,0,0,0,0,0
1,order_items.order_id -> orders.order_id,parent,orders.csv,order_id,0,0,0,0,0,0
2,returns.product_id -> products.product_id,child,returns.csv,product_id,0,0,0,0,0,0
3,returns.product_id -> products.product_id,parent,products.csv,product_id,0,0,0,0,0,0
4,customers.zip -> geography.zip,child,customers.csv,zip,0,0,0,0,0,0
5,customers.zip -> geography.zip,parent,geography.csv,zip,0,0,0,0,0,0
6,orders.zip -> geography.zip,child,orders.csv,zip,0,0,0,0,0,0
7,orders.zip -> geography.zip,parent,geography.csv,zip,0,0,0,0,0,0


## 6. Key observations and warnings

Print a concise summary of missing files, missing key columns, orphan keys, suspicious missing key values, and inconsistent formatting.

In [7]:
orphan_warning_df = key_integrity_summary_df[
    key_integrity_summary_df["unique_orphan_keys"].fillna(0) > 0
].copy()

missing_key_value_warning_df = formatting_diagnostics_df[
    formatting_diagnostics_df["normalized_missing_count"].fillna(0) > 0
].copy()

formatting_warning_df = formatting_diagnostics_df[
    (formatting_diagnostics_df["values_with_outer_whitespace"].fillna(0) > 0)
    | (formatting_diagnostics_df["empty_strings_after_strip"].fillna(0) > 0)
    | (formatting_diagnostics_df["decimal_integer_string_count"].fillna(0) > 0)
    | (formatting_diagnostics_df["leading_zero_string_count"].fillna(0) > 0)
].copy()

print("Warning summary")
print("- Missing CSV files:", missing_files_df["file"].tolist() if len(missing_files_df) else "None")
print(
    "- Missing key columns:",
    missing_key_columns_df[["file", "logical_key", "available_columns"]].to_dict("records") if len(missing_key_columns_df) else "None",
)
print(
    "- Detected orphan keys:",
    orphan_warning_df[["check_name", "unique_orphan_keys", "orphan_key_value_count", "orphan_unique_pct", "orphan_value_pct"]].to_dict("records") if len(orphan_warning_df) else "None",
)
print(
    "- Key columns with suspicious missing values:",
    missing_key_value_warning_df[["check_name", "side", "file", "column", "normalized_missing_count"]].to_dict("records") if len(missing_key_value_warning_df) else "None",
)
print(
    "- Key columns with inconsistent formatting:",
    formatting_warning_df[["check_name", "side", "file", "column", "values_with_outer_whitespace", "empty_strings_after_strip", "decimal_integer_string_count", "leading_zero_string_count"]].to_dict("records") if len(formatting_warning_df) else "None",
)

print("\nMissing files")
display(missing_files_df if len(missing_files_df) else pd.DataFrame({"message": ["No expected CSV files are missing."]}))

print("\nMissing key columns")
display(missing_key_columns_df if len(missing_key_columns_df) else pd.DataFrame({"message": ["All required key columns were resolved."]}))

print("\nOrphan key warnings")
display(orphan_warning_df if len(orphan_warning_df) else pd.DataFrame({"message": ["No orphan keys detected in required checks."]}))

print("\nMissing key value warnings")
display(missing_key_value_warning_df if len(missing_key_value_warning_df) else pd.DataFrame({"message": ["No missing key values detected after normalization."]}))

print("\nFormatting warnings")
display(formatting_warning_df if len(formatting_warning_df) else pd.DataFrame({"message": ["No suspicious key formatting detected."]}))

Warning summary
- Missing CSV files: None
- Missing key columns: None
- Detected orphan keys: None
- Key columns with suspicious missing values: None
- Key columns with inconsistent formatting: None

Missing files


,message
0,No expected CSV files are missing.



Missing key columns


,message
0,All required key columns were resolved.



Orphan key warnings


,message
0,No orphan keys detected in required checks.



Missing key value warnings


,message
0,No missing key values detected after normalization.



Formatting warnings


,message
0,No suspicious key formatting detected.
